In [ ]:
from bctools.io import InstrumentResponse
from bctools.loc import LocalLocTable
from bctools.spectra.spectrum import BandFunction,Comptonized
import os
import math
import multiprocessing
import itertools
import pickle 
import numpy as np
from bctools.loc import TSMap, NormLocLike
import astropy.units as u
from astropy.coordinates import SkyCoord
import matplotlib.pyplot as plt
import healpy as hp
import time



new_scheme = True

if new_scheme:
   run_name = "run10_newschema"
else:
   run_name = "run10"
irf_path = "/data/models/irf_summed_"+run_name+".h5"
dir_path = "/data/test_newrepo/"

In [ ]:
# Convolve an full instrument response with a hypothetical spectrum
#irf_path = "/data/models/irf_summed.h5"
#irf_path = "/data/models/run_1_noeffect_irf_summed.h5"
import pickle

load_from_file = True

if load_from_file:

    with open('/data/models/LUTS/soft_lut_' + run_name + '.pkl', 'rb') as f:
        soft_sky_loctable = pickle.load(f)
    
    with open('/data/models/LUTS/medium_lut_' + run_name + '.pkl', 'rb') as f:
        medium_sky_loctable = pickle.load(f)
    
    with open('/data/models/LUTS/hard_lut_' + run_name + '.pkl', 'rb') as f:
        hard_sky_loctable = pickle.load(f)
    
else:

    with InstrumentResponse(irf_path) as irf: 
        
        # Hypothetical spectrum
        # This normalization corresponds to 1 ph/cm2/s between 50-300 keV
        #spectrum = PowerLaw(60,2)
        #spectrum = PowerLaw._from_megalib(['PowerLaw',10,10000,2],"5.0")
        soft_spectrum = BandFunction._from_megalib(['BandFunction',10,10000,-1.9,-3.7,230],"10.0")
        medium_spectrum = BandFunction._from_megalib(['BandFunction',10,10000,-1,-2.3,699.9],"10.0")
        hard_spectrum = Comptonized._from_megalib(['Comptonized',10,10000,-0.5,1500],"10.0")
        
        # In this case we integrate the rate from all energy channels. 
        # You can subdivide the data into multiple energy channel groups
        soft_local_loctable = LocalLocTable.from_irf(irf, soft_spectrum,energy_channels = 1) # [80,2000]
        medium_local_loctable = LocalLocTable.from_irf(irf, medium_spectrum,energy_channels = 1)
        hard_local_loctable = LocalLocTable.from_irf(irf, hard_spectrum,energy_channels = 1)
        
    # The local_loctable contains the expected rates in spacecraft coordinates
    # We now need to use this to estimate the total expected counts in sky coordinate for
    # the full duration of an event. 
    # In this case we simply have a 1 second event and specifying the attitude by a quaternion
    # ([0,0,0,1] corresponds to the identity rotation). You can have multiple attitude-duration
    # pairs to correctly model long duration events.
    soft_sky_loctable = soft_local_loctable.to_skyloctable(attitude = [0,0,0,1], duration = 1)
    medium_sky_loctable = medium_local_loctable.to_skyloctable(attitude = [0,0,0,1], duration = 1)
    hard_sky_loctable = hard_local_loctable.to_skyloctable(attitude = [0,0,0,1], duration = 1)
    
    #Store LUT
    
    import pickle
    
    # Salvataggio su file
    with open('/data/models/LUTS/soft_lut_'+run_name+'.pkl', 'wb') as f:
        pickle.dump(soft_sky_loctable, f)
    with open('/data/models/LUTS/medium_lut_'+run_name+'.pkl', 'wb') as f:
        pickle.dump(medium_sky_loctable, f)
    with open('/data/models/LUTS/hard_lut_'+run_name+'.pkl', 'wb') as f:
        pickle.dump(hard_sky_loctable, f)

In [ ]:

def angular_distance(theta1, phi1, theta2, phi2):
    """
    Compute the angular (great-circle) distance in degrees between two points
    specified by (theta, phi) coordinates in degrees.

    Parameters:
        theta1, phi1: floats, coordinates of the first point (degrees).
        theta2, phi2: floats, coordinates of the second point (degrees).

    Returns:
        angular_dist_deg: float, angular distance between the two points in degrees.
    """
    # Adjust theta values by shifting -90
    theta1 = 90 - theta1
    theta2 = 90 - theta2

    # Convert angles to radians
    theta1_rad = math.radians(theta1)
    phi1_rad = math.radians(phi1)
    theta2_rad = math.radians(theta2)
    phi2_rad = math.radians(phi2)
    
    # Compute the difference in longitude
    delta_phi = abs(phi1_rad - phi2_rad)

    # Apply spherical law of cosines
    value = (math.sin(theta1_rad) * math.sin(theta2_rad) +
             math.cos(theta1_rad) * math.cos(theta2_rad) * math.cos(delta_phi))
    
    # Clamp value to [-1, 1] to avoid floating point errors in acos
    value_clamped = max(-1.0, min(1.0, value)) 
    
    angular_dist = math.acos(value_clamped)
    
    # Convert angular distance from radians to degrees
    angular_dist_deg = math.degrees(angular_dist)
    
    return angular_dist_deg

In [ ]:
print(f"Soft Look-up tables: {soft_sky_loctable.labels}")
print(f"Medium Look-up tables: {medium_sky_loctable.labels}")
print(f"Hard Look-up tables: {hard_sky_loctable.labels}")



In [ ]:
medium_sky_loctable.get_expectation_map('BGO_Z1').data

In [ ]:
import pickle
if False:

    # Salva l'array in un file usando pickle
    with open("/data/exp_medium_Y1.pkl", "wb") as f:
        pickle.dump(medium_sky_loctable.get_expectation_map('BGO_Y1').data, f)


In [ ]:

def get_coord_helpix(nside,pix_id):
    
    # Parametri
    nside = 16  # Sostituisci con il tuo valore di NSIDE
    ipix = 1   # Sostituisci con l'ID del pixel HEALPix (in modalità nested)
    
    # Ottieni le coordinate angolari (theta, phi) del centro del pixel
    theta, phi = hp.pix2ang(nside, ipix, nest=True)
    
    # Converti in coordinate equatoriali (RA, Dec)
    ra = phi * 180.0 / np.pi      # phi è la longitudine in radianti (RA)
    dec = 90.0 - theta * 180.0 / np.pi  # theta è la colatitudine (Dec)
    
    # Crea l'oggetto SkyCoord
    coord = SkyCoord(ra=ra * u.deg, dec=dec * u.deg, frame='icrs')

    return coord


In [ ]:
medium_sky_loctable.get_expectation_map('BGO_Z1').plot();

In [ ]:

if new_scheme:
    file_name_test =  "run63_10x_newschema_shared"
else:
        
    file_name_test = "run63_medium_10fact_mega_shared"

file_path =dir_path+'/'+file_name_test+'_dataset.pkl'

# Load the array from the pickle file
with open(file_path, 'rb') as file:
        loaded_array_test = pickle.load(file)


In [ ]:
loaded_array_test.shape

In [ ]:
for grb in loaded_array_test:
    print(grb)
    break

In [ ]:
import numpy as np

filter_flux = 0
filter_spectra = 0

filtered_loaded_array_test = np.empty(loaded_array_test.shape[0], dtype=object)

count = 0

if filter_flux == 1:
    for grb in loaded_array_test:
        if grb['flux'] > 10 and grb['flux'] <= 15:
            filtered_loaded_array_test[count] = grb
            count = count+1

    loaded_array_test = filtered_loaded_array_test[:count]

filtered_loaded_array_test = np.empty(loaded_array_test.shape[0], dtype=object)

count = 0

if filter_spectra==1:
    for grb in loaded_array_test:
        #if  grb['spectra'] == 'medium':
        if  '1500' in grb['spectrum']:
            filtered_loaded_array_test[count] = grb
            count = count+1

    loaded_array_test = filtered_loaded_array_test[:count]

In [ ]:
loaded_array_test.shape

In [ ]:
test_dataset=loaded_array_test

In [ ]:
test_dataset[0]

In [ ]:
b_sim = np.array([57.6053, 58.7157, 51.4131, 48.2891, 47.7293, 45.9617])
random_bkg = np.random.poisson(np.array([b_sim[3],b_sim[2],b_sim[5],b_sim[4],b_sim[1],b_sim[0]])*20)

def process_lut_source(grb_list,random_bkg):

    distance_list = []
    conf_area_list = []
    
    for grb in grb_list:
        
        theta_real = float(grb['coord'][0])
        phi_real = float(grb['coord'][1])
        
        s_counts = np.array([grb['counts'][3],grb['counts'][2],grb['counts'][5],grb['counts'][4],grb['counts'][1],grb['counts'][0]])

        b_sim = np.array([57.6053, 58.7157, 51.4131, 48.2891, 47.7293, 45.9617])
        random_bkg = np.random.poisson(np.array([b_sim[3],b_sim[2],b_sim[5],b_sim[4],b_sim[1],b_sim[0]])*20)
        
        s_counts = s_counts+random_bkg
        b_counts = np.array([b_sim[3],b_sim[2],b_sim[5],b_sim[4],b_sim[1],b_sim[0]])*20
        
        medium_sqrt_ts, mdium_ra_loc, medium_dec_loc, medium_cont_radius,medium_ts,medium_loc_tsvalue,medium_cont_area = localize_grb(medium_sky_loctable,s_counts, b_counts,theta_real,phi_real)
       
        max_ts = medium_sqrt_ts
       
        if max_ts == medium_sqrt_ts:
            original_ts_value = medium_loc_tsvalue
            best_ts = medium_ts
            ra_loc=mdium_ra_loc
            dec_loc=medium_dec_loc
            cont_radius = medium_cont_radius
            cont_area = medium_cont_area
       
        theta_loc, phi_loc = ra_dec_to_theta_phi(ra_loc, dec_loc)
    
        dist = angular_distance(theta_loc, phi_loc, theta_real, phi_real)
        theta_dist = np.abs(theta_loc - theta_real)
        phi_dist = diff_phi(phi_loc, phi_real)

        distance_list.append(dist)
        conf_area_list.append(cont_area)

    distance_avg = np.mean(distance_list)
    cont_area_avg = np.mean(conf_area_list)
    
    result = [distance_avg,cont_area_avg]
    
    return result
    

def localize_grb(sky_loctable,s,b,theta_real,phi_real):

    
    sky_loctable.set_background(b)
    sky_loctable.set_data(s)

    # Define a map of nside = 32. Note that this is a finer resolution
    #that the underlying look-up table, which will be interpolated
    ts = TSMap(nside = 32, coordsys = 'icrs')

    # NormLocLike is a subclass of LocLike and computes
    # a Poisson likelihood for counting instruments. The
    # overall normalization is the only free parameter
    norm_likelihood = NormLocLike(sky_loctable)
   
    # Compute the TS map from one or more LocLikelihood
    ts.compute(norm_likelihood)
    #print("#"+str(theta_real)+"#")
    
    # Correggi theta_real prima di convertirlo
    if theta_real < 0:
        print(f"Invalid theta_real: {theta_real}")
        theta_real = 0
    elif theta_real > 180:
        print(f"Invalid theta_real: {theta_real}")
        theta_real = 180
    
    theta_rad = np.deg2rad(theta_real)
    phi_rad = np.deg2rad(phi_real)
    
    if theta_rad < 0 or theta_rad > np.pi:
        print(f"theta_rad fuori range: {theta_rad}")


    #print(dir(ts))
    ipix = ts.ang2pix(theta_rad, phi_rad)

    # Get the value at the given coordinates
    #ipix = ts.ang2pix(ts.nside, theta_real * u.deg.to(u.rad), phi_real * u.deg.to(u.rad))
    original_ts_value = ts._data[ipix]

    # Inizializza l'array dei livelli di confidenza
    confidence_levels = np.arange(0, 1.01, 0.01)
    
    # Calcola i raggi di contenimento per ciascun valore di cont
    containment_radius = np.array([
        np.sqrt(ts.error_area(cont=cont) / np.pi).to(u.deg).value
        for cont in confidence_levels
    ])
  
    #original_ts_value = -1
    cont_area = ts.error_area(cont = .9).to(u.deg**2).value

    

    return np.max(ts),ts.best_loc().ra.deg,ts.best_loc().dec.deg,containment_radius,ts,original_ts_value,cont_area

def ra_dec_to_theta_phi(ra, dec):

    theta = 90-dec
    
    phi = ra
    
    return theta, phi

def diff_phi(a1, a2):
    
    diff = abs(a1 - a2)
    
    if diff > 180:
        diff = 360 - diff
    
    return diff

In [ ]:
results_bkg = []
spectra_fitted = 0


def init_worker():
    seed = int.from_bytes(os.urandom(4), "little")
    np.random.seed(seed)


def process_in_parallel(test_dataset,random_bkg):
    with multiprocessing.Pool(processes=200, initializer=init_worker) as pool:
        results = pool.starmap(process_lut_source, zip(test_dataset, itertools.repeat(random_bkg)))
    return results


results_bkg = process_in_parallel(test_dataset,random_bkg)


In [ ]:
distances = []
area_array = []

for r in results_bkg:
    distances.append(r[0])
    area_array.append(r[1])

distances = np.array(distances)
area_array = np.array(area_array)


In [ ]:
plt.hist(distances)

In [ ]:
import healpy as hp
import numpy as np
import matplotlib.pyplot as plt
import pickle
hp.projview(
    np.array(area_array),
    coord=["G"],
    projection_type="aitoff",          # Cambiato da "mollweide" a "aitoff"
    graticule=True,
    graticule_labels=True,
    longitude_grid_spacing=60,
    title=file,
    latitude_grid_spacing=30,
    cmap="turbo",
    nest=True,
    unit="", 
    fontsize={
        "xlabel": 14,
        "ylabel": 14,
        "title": 16,
        "xtick_label": 14,
        "ytick_label": 14,
        "cbar_label": 14,
        "cbar_tick_label": 14  # qui imposti il font size dei numeri della colorbar
    },
    override_plot_properties={
        "cbar_shrink": 0.8,
        "cbar_pad": 0.05,
        "cbar_label_pad": 5
    }
    
)
plt.show()

In [ ]:
import pickle
if True:

    # Salva l'array in un file usando pickle
    with open("/data/bc_"+file_name_test+"_distall.pkl", "wb") as f:
        pickle.dump(distances, f)
        
    # Salva l'array in un file usando pickle
    with open("/data/bc_"+file_name_test+"_cont_area.pkl", "wb") as f:
        pickle.dump(area_array, f)

